# Notebook 5: Who Controls the Shadow?

**Act II — Power** | **Duration:** 55 min | **Register:** Semi-Formal

---

Notebooks 1–4 treated $y$ as a single vector. Now zoom in: each element $y_i$ contributes differently to the projection. Some observations "pull" the shadow harder than others.

Imagine a tug-of-war. Every observation is pulling the shadow toward itself. But some observations have longer arms — they’re standing further from the center of the data, so they exert more leverage on where the shadow falls. The hat matrix diagonal tells you each observation’s arm length.

**Core theorems:** Leverage ($h_{ii}$) and influence (Cook’s distance) as geometric concepts.

**What you’ll need:**
- `regression_geometry.core` — the projection engine
- `regression_geometry.plots` — static visualizations
- `regression_geometry.data` — the Meridian dataset

*"Not all hands pull with equal force on the rope."*

In [ ]:
# ============================================================
# Notebook 05: "Who Controls the Shadow?"
# Regression from the Inside: Seeing the Geometry of Linear Models
# ============================================================

# --- Environment setup (run this cell first) ---
import sys

# Install regression_geometry package if not available
try:
    import regression_geometry
except ImportError:
    # Running in Colab or fresh environment — install from GitHub
    print("Installing regression_geometry package...")
    !pip install -q git+https://github.com/YOUR_USERNAME/regression-geometry-course.git
    print("Done! If you see import errors below, restart the runtime (Runtime → Restart) and run this cell again.")

# --- Standard imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import OLSInfluence

from regression_geometry.core import ColumnSpace, Projection, HatMatrix, demean
from regression_geometry.data import load_meridian
from regression_geometry.colors import *

# --- Rendering backend toggle ---
INTERACTIVE = True
try:
    from regression_geometry import interactive as viz_mod
    if not viz_mod.AVAILABLE:
        INTERACTIVE = False
except ImportError:
    INTERACTIVE = False

if INTERACTIVE:
    from regression_geometry import interactive as viz
else:
    from regression_geometry import plots as viz

from regression_geometry.scoreboard import GeometricScoreboard

# --- Reproducibility ---
np.random.seed(42)

In [ ]:
# Load the Meridian dataset
mer = load_meridian()
y = mer["salary"].values
print(f"Meridian Analytics: {len(mer)} employees")
mer.head()

## Beat 1: From Projection to Individual Observations

The hat matrix $H$ maps $y$ to $\hat{y}$: each fitted value $\hat{y}_i = \sum_j h_{ij} y_j$ is a weighted combination of all observations. The diagonal element $h_{ii}$ tells you how much observation $i$ influences its **own** fitted value.

The $h_{ii}$ values live between $1/n$ and $1$. An observation with $h_{ii}$ near $1$ is pulling the shadow almost entirely toward itself.

In [ ]:
# Full model: salary ~ experience + education + gender
X_full = mer[["experience", "education", "gender"]].values
cs_full = ColumnSpace(X_full, add_intercept=True)
proj_full = cs_full.project(y)

# Compute the hat matrix
hm = HatMatrix(cs_full.hat_matrix())
h_diag = hm.diagonal()

print(f"Number of observations: {cs_full.n}")
print(f"Number of parameters:   {cs_full.p}")
print(f"Average leverage:       {hm.average_leverage():.4f} = p/n = {cs_full.p}/{cs_full.n}")
print(f"Min h_ii:               {h_diag.min():.6f}")
print(f"Max h_ii:               {h_diag.max():.6f}")
print()
print("Top 5 highest-leverage observations:")
top5 = np.argsort(h_diag)[-5:][::-1]
for idx in top5:
    row = mer.iloc[idx]
    print(f"  Obs {idx:>4}: h_ii={h_diag[idx]:.4f}, exp={row['experience']:.0f}, "
          f"edu={row['education']:.0f}, salary=${row['salary']:,.0f}")

## Beat 2: Leverage as Geometric Distance

The hat value $h_{ii}$ measures how far observation $i$ is from the centroid of the predictor space. Observations with extreme predictor values — far from the center — have long "arms" in the tug-of-war.

### 🛑 PREDICT FIRST

Looking at the Meridian dataset, which employee do you think has the **highest leverage**?
- The CEO (32 years experience, salary ~$2.1M)?
- A brand new hire (0 years experience)?
- Someone else?

*Write your prediction before running the next cell.*

In [ ]:
# Who has the highest leverage?
max_lev_idx = np.argmax(h_diag)
max_lev_row = mer.iloc[max_lev_idx]

print(f"Highest-leverage observation: index {max_lev_idx}")
print(f"  Experience: {max_lev_row['experience']:.0f} years")
print(f"  Education:  {max_lev_row['education']:.1f} years")
print(f"  Salary:     ${max_lev_row['salary']:,.0f}")
print(f"  h_ii:       {h_diag[max_lev_idx]:.4f}")
print()
print(f"Threshold for 'high leverage': 2p/n = {2 * cs_full.p / cs_full.n:.4f}")
print(f"This observation's leverage is {h_diag[max_lev_idx] / (2 * cs_full.p / cs_full.n):.1f}x the threshold.")

In [ ]:
# Leverage stem plot
viz.plot_leverage(hm, highlight_indices=[max_lev_idx])

## Beat 3: Leverage vs. Influence — The Critical Distinction

High leverage means an observation **can** distort the projection. But **does** it? That depends on whether the observation is also unusual in the $y$-direction — whether it has a large residual.

**Cook’s distance** combines both:

$$D_i \propto h_{ii} \times e_i^2$$

An observation needs **both** a long arm (high leverage) **and** a strong pull (large residual) to actually influence the results.

**Influence = leverage × surprise.** An observation that fits the trend is harmless no matter how extreme its predictors.

In [ ]:
# Compute Cook's distance
e = proj_full.residuals
mse = np.sum(e**2) / (cs_full.n - cs_full.p)
cooks_d = hm.cooks_distance(e, mse, cs_full.p)

# The CEO (highest leverage)
print(f"CEO (obs {max_lev_idx}):")
print(f"  Leverage h_ii = {h_diag[max_lev_idx]:.4f} (VERY high)")
print(f"  Residual e_i  = ${e[max_lev_idx]:,.0f}")
print(f"  Cook's D      = {cooks_d[max_lev_idx]:.4f}")
print()

# Find the observation with the highest Cook's D
max_cook_idx = np.argmax(cooks_d)
max_cook_row = mer.iloc[max_cook_idx]
print(f"Highest Cook's D observation (obs {max_cook_idx}):")
print(f"  Experience: {max_cook_row['experience']:.0f} years")
print(f"  Education:  {max_cook_row['education']:.1f} years")
print(f"  Salary:     ${max_cook_row['salary']:,.0f}")
print(f"  Leverage h_ii = {h_diag[max_cook_idx]:.4f}")
print(f"  Residual e_i  = ${e[max_cook_idx]:,.0f}")
print(f"  Cook's D      = {cooks_d[max_cook_idx]:.4f}")

In [ ]:
# Influence diagram: leverage vs. squared residual, point size = Cook's D
viz.plot_influence_diagram(
    hm, proj_full.residuals, mse, cs_full.p,
    highlight_indices=[max_lev_idx, max_cook_idx]
)

## Beat 4: The Workflow Trap — The Wrong Outlier

The CEO’s leverage is off the charts. Standard practice says to check whether removing high-leverage observations changes your results. Let’s try it.

In [ ]:
# Remove the CEO (highest-leverage observation) and re-run
mask_no_ceo = np.ones(len(mer), dtype=bool)
mask_no_ceo[max_lev_idx] = False

X_no_ceo = X_full[mask_no_ceo]
y_no_ceo = y[mask_no_ceo]

model_full_orig = sm.OLS(y, sm.add_constant(X_full)).fit()
model_no_ceo = sm.OLS(y_no_ceo, sm.add_constant(X_no_ceo)).fit()

labels = ["intercept", "experience", "education", "gender"]
print("Coefficients with and without the CEO:")
print(f"{'Variable':<15} {'With CEO':>12} {'Without CEO':>12} {'Change':>10}")
print("-" * 51)
for i, lab in enumerate(labels):
    diff = model_no_ceo.params[i] - model_full_orig.params[i]
    pct = 100 * diff / abs(model_full_orig.params[i]) if model_full_orig.params[i] != 0 else 0
    print(f"{lab:<15} {model_full_orig.params[i]:>12,.2f} {model_no_ceo.params[i]:>12,.2f} {pct:>9.1f}%")

print()
print("Coefficients barely changed. The CEO looks harmless.")

### 🛑 PREDICT FIRST

You’ve removed the highest-leverage observation and nothing changed. **Does that mean no single observation is influencing your results?**

*Write your prediction before running the next cell.*

In [ ]:
# The reveal: look at Cook's distance, not leverage alone
top_cook_indices = np.argsort(cooks_d)[-5:][::-1]

print("Top 5 observations by Cook's distance:")
print(f"{'Obs':>5} {'h_ii':>8} {'|e_i|':>12} {'Cook D':>10} {'Exp':>5} {'Edu':>5} {'Salary':>12}")
print("-" * 60)
for idx in top_cook_indices:
    row = mer.iloc[idx]
    print(f"{idx:>5} {h_diag[idx]:>8.4f} {abs(e[idx]):>12,.0f} {cooks_d[idx]:>10.4f} "
          f"{row['experience']:>5.0f} {row['education']:>5.0f} ${row['salary']:>11,.0f}")

print()
print(f"The CEO (obs {max_lev_idx}) has HUGE leverage but low Cook's D.")
print(f"Observation {max_cook_idx} has moderate leverage but HIGH Cook's D.")

In [ ]:
# Now remove the high-influence observation and see the real damage
mask_no_inf = np.ones(len(mer), dtype=bool)
mask_no_inf[max_cook_idx] = False

X_no_inf = X_full[mask_no_inf]
y_no_inf = y[mask_no_inf]

model_no_inf = sm.OLS(y_no_inf, sm.add_constant(X_no_inf)).fit()

print(f"Removing obs {max_cook_idx} (highest Cook's D):")
print(f"{'Variable':<15} {'Full model':>12} {'Without obs':>12} {'Change':>10}")
print("-" * 51)
for i, lab in enumerate(labels):
    diff = model_no_inf.params[i] - model_full_orig.params[i]
    pct = 100 * diff / abs(model_full_orig.params[i]) if model_full_orig.params[i] != 0 else 0
    print(f"{lab:<15} {model_full_orig.params[i]:>12,.2f} {model_no_inf.params[i]:>12,.2f} {pct:>9.1f}%")

print()
print("THIS observation moves the coefficients — especially education.")
print("You missed it because you were looking at leverage alone, not influence.")

⚠️ **WORKFLOW TRAP**

You looked at leverage — arm length — and removed the obvious outlier. But **influence is arm length TIMES pull.** The CEO had a long arm but wasn’t pulling. The high-Cook’s observation had a shorter arm but was yanking the education coefficient hard.

Leverage tells you who **could** matter. Influence tells you who **does**.

## Beat 5: Exploring Leverage and Influence Together

In [ ]:
# Three key observations:
# 1. CEO — high leverage, low influence
# 2. High Cook's D obs — moderate leverage, high influence
# 3. A typical observation — low both
typical_idx = np.argmin(np.abs(h_diag - hm.average_leverage()))

print("Three observations, three stories:")
print(f"{'':<20} {'CEO':>12} {'Influential':>12} {'Typical':>12}")
print(f"{'Observation':<20} {max_lev_idx:>12} {max_cook_idx:>12} {typical_idx:>12}")
print(f"{'h_ii':<20} {h_diag[max_lev_idx]:>12.4f} {h_diag[max_cook_idx]:>12.4f} {h_diag[typical_idx]:>12.4f}")
print(f"{'|e_i|':<20} {abs(e[max_lev_idx]):>12,.0f} {abs(e[max_cook_idx]):>12,.0f} {abs(e[typical_idx]):>12,.0f}")
print(f"{'Cook D':<20} {cooks_d[max_lev_idx]:>12.4f} {cooks_d[max_cook_idx]:>12.4f} {cooks_d[typical_idx]:>12.4f}")

In [ ]:
# Cook's distance stem plot with both key observations highlighted
viz.plot_cooks_distance(cooks_d, highlight_indices=[max_lev_idx, max_cook_idx])

### 🛑 DIAGNOSE FIRST

Looking at the Cook’s distance plot above, identify:

1. How many observations have Cook’s D above the threshold line?
2. Is the CEO among them?
3. Which observation has the highest influence, and **why** (in terms of leverage and residual)?

*Write your answers, then verify with the code below.*

In [ ]:
# Verify
threshold = 4 / cs_full.n  # common Cook's D threshold
n_above = np.sum(cooks_d > threshold)

print(f"1. Observations with Cook's D > {threshold:.4f}: {n_above}")
status = "above" if cooks_d[max_lev_idx] > threshold else "below"
print(f"2. CEO's Cook's D = {cooks_d[max_lev_idx]:.4f} — {status} threshold")
print(f"3. Highest Cook's D: obs {max_cook_idx} (D = {cooks_d[max_cook_idx]:.4f})")
print(f"   h_ii = {h_diag[max_cook_idx]:.4f}, |e_i| = ${abs(e[max_cook_idx]):,.0f}")
print("   High influence because it has BOTH moderate leverage AND a large residual.")

🔗 **Back to Statsmodels**

Statsmodels provides all these diagnostics through `OLSInfluence`.

In [ ]:
# statsmodels influence diagnostics
model_sm = sm.OLS(y, sm.add_constant(X_full)).fit()
influence = OLSInfluence(model_sm)

# Compare our computations with statsmodels
h_diag_sm = influence.hat_matrix_diag
cooks_d_sm = influence.cooks_distance[0]

print("Our computation vs. statsmodels:")
print(f"  h_ii match?    {np.allclose(h_diag, h_diag_sm)}")
print(f"  Cook's D match? {np.allclose(cooks_d, cooks_d_sm, atol=1e-6)}")
print()
print("Statsmodels equivalents:")
print("  h_ii     → OLSInfluence(model).hat_matrix_diag")
print("  Cook's D → OLSInfluence(model).cooks_distance[0]")
print("  Influence plot → OLSInfluence(model).plot_influence()")

✍️ **The Memo**

Write a memo (3 sentences) to Meridian’s Head of Data Science explaining why you are **not** concerned about the CEO’s extreme salary distorting the regression, but you **are** concerned about a specific observation with high influence.

**Forbidden words:** *leverage, hat matrix, Cook’s distance, influence.* Explain using the concept of "pull on the results."

*Your memo here:*

...

## Geometric Scoreboard

The final gauge is now unlocked: **κ** (condition number). All five gauges are now active. For the rest of the course, you can read the full Scoreboard.

In [ ]:
# Geometric Scoreboard — all 5 gauges unlocked!
viz.plot_scoreboard(
    proj_full, cs_full,
    active_gauges=["theta", "r_squared", "leverage", "residual_norm", "kappa"]
)

---
### Summary

**What you learned:**

- **Leverage is arm length in the column space.** The hat matrix diagonal $h_{ii}$ measures how far observation $i$ is from the center of the predictor space.
- **Influence is arm length times pull.** An observation needs both high leverage AND a large residual to actually move the results. Cook's distance captures both.
- **High leverage doesn't mean high influence.** The CEO had the longest arm but wasn't pulling against the crowd. The real damage came from an observation with a shorter arm but a much stronger tug.

**Key geometric insight:** ***Influence = leverage × surprise. An observation that fits the trend is harmless no matter how extreme its predictors.***

### Next

What happens when the column space itself becomes unreliable? Notebook 6 asks what happens when the geometry you've learned to trust **betrays you** — when the predictors are so correlated that the column space collapses from a plane into something dangerously thin.

---

## Summary and Bridge

**What you learned:**

- **Leverage is arm length in the column space.** The hat matrix diagonal $h_{ii}$ measures how far observation $i$ is from the center of the predictor space.
- **Influence is arm length times pull.** An observation needs both high leverage AND a large residual to actually move the results. Cook’s distance captures both.
- **High leverage doesn’t mean high influence.** The CEO had the longest arm but wasn’t pulling against the crowd. The real damage came from an observation with a shorter arm but a much stronger tug.

**Key geometric insight:** *Influence = leverage × surprise. An observation that fits the trend is harmless no matter how extreme its predictors.*

---

**Next:** What happens when the column space itself becomes unreliable? Notebook 6 asks what happens when the geometry you’ve learned to trust **betrays you** — when the predictors are so correlated that the column space collapses from a plane into something dangerously thin.